# Memory System Demo

This notebook exercises the memory architecture described in `memory.md` against the current implementation in `src/memory`.

It covers:
- semantic memory in warm storage
- episodic memory across hot and warm layers
- archival from warm to cold and rehydration
- local to global retrieval escalation
- sleeping task queue behavior
- working and procedural memory examples


In [1]:
from __future__ import annotations

import sys
from datetime import timedelta
from pathlib import Path
from tempfile import TemporaryDirectory


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root={REPO_ROOT}")


repo_root=/Users/saketm10/Projects/openclaw_agents


In [2]:
from src.memory.layers import ColdMemoryLayer, HotMemoryLayer, WarmMemoryLayer
from src.memory.models import RetrievalContext, SleepingTask, utc_now
from src.memory.retrieval.retriever import LayeredMemoryRetriever
from src.memory.router import MemoryRouter
from src.memory.service import MemoryService
from src.memory.store import MemoryStore
from src.memory.tasks.sleeping_queue import SleepingTaskQueue
from src.memory.types import EpisodicMemory, ProceduralMemory, SemanticMemory, WorkingMemory


def build_store(root: Path, name: str, *, scope: str, agent_id: str | None) -> MemoryStore:
    return MemoryStore(
        hot_layer=HotMemoryLayer(max_items=8),
        warm_layer=WarmMemoryLayer(root / f"{name}.duckdb"),
        cold_layer=ColdMemoryLayer(root / f"{name}.jsonl"),
        archive_after_days=30,
        default_scope=scope,
        agent_id=agent_id,
    )


tmp = TemporaryDirectory()
MEMORY_ROOT = Path(tmp.name)
local_store = build_store(MEMORY_ROOT, "local", scope="agent_local", agent_id="mailmind")
global_store = build_store(MEMORY_ROOT, "global", scope="global", agent_id=None)
service = MemoryService(
    local_store=local_store,
    global_store=global_store,
    router=MemoryRouter(
        local_retriever=LayeredMemoryRetriever(local_store),
        global_retriever=LayeredMemoryRetriever(global_store),
    ),
)

print(f"memory_root={MEMORY_ROOT}")


memory_root=/var/folders/x3/r36gzhhj13b6sbqsg2hgn0zm0000gn/T/tmp10wd7kdk


In [3]:
# Semantic memory should land in warm storage and be retrievable.
semantic = service.add_local(
    SemanticMemory(
        content={"fact": "user prefers research roles at DeepMind"},
        metadata={"tags": ["preference"]},
    )
)

results = service.retrieve(
    "DeepMind",
    filters={"type": "semantic", "agent_id": "mailmind"},
    limit=5,
)

print({
    "stored_type": semantic.type,
    "stored_layer": semantic.layer,
    "stored_scope": semantic.scope,
    "stored_agent_id": semantic.agent_id,
})
print([record.content_text for record in results])


{'stored_type': 'semantic', 'stored_layer': 'warm', 'stored_scope': 'agent_local', 'stored_agent_id': 'mailmind'}
["{'fact': 'user prefers research roles at DeepMind'}"]


In [4]:
# Episodic memory should be searchable and present in both hot cache and warm storage.
episodic = local_store.add(EpisodicMemory(content={"event": "classified a DeepMind recruiting email"}))

print({
    "stored_layer": episodic.layer,
    "hot_hit": local_store.hot_layer.get(episodic.id) is not None,
    "warm_hit": local_store.warm_layer.get(episodic.id) is not None,
})


{'stored_layer': 'warm', 'hot_hit': True, 'warm_hit': True}


In [5]:
# Old warm memory should archive to cold JSONL and rehydrate into warm/hot on access.
archived_candidate = SemanticMemory(
    content={"fact": "old archived preference"},
    created_at=utc_now() - timedelta(days=45),
    metadata={"tags": ["archival"]},
).store_warm(local_store)

archived_count = local_store.archive_old()
cold_hit = local_store.cold_layer.get(archived_candidate.id)
warm_before_rehydrate = local_store.warm_layer.get(archived_candidate.id)
rehydrated = local_store.get(archived_candidate.id)

print({
    "archived_count": archived_count,
    "cold_hit": cold_hit is not None,
    "warm_before_rehydrate": warm_before_rehydrate is not None,
    "rehydrated_layer": rehydrated.layer if rehydrated else None,
    "rehydrated_hot": local_store.hot_layer.get(archived_candidate.id) is not None,
})

print("cold archive contents:")
print((MEMORY_ROOT / "local.jsonl").read_text(encoding="utf-8"))


{'archived_count': 1, 'cold_hit': True, 'warm_before_rehydrate': False, 'rehydrated_layer': 'warm', 'rehydrated_hot': True}
cold archive contents:
{"id":"b4e1a80e-ef3c-4792-b785-dd943d9d34ad","agent_id":"mailmind","scope":"agent_local","type":"semantic","layer":"cold","content":{"fact":"old archived preference"},"content_text":"{'fact': 'old archived preference'}","content_json":{"fact":"old archived preference"},"source_type":"system","source_id":null,"tags":["archival"],"metadata":{"agent":"mailmind","agent_id":"mailmind","confidence":null,"importance":null,"priority":"medium","scope":"agent_local","source":"system","source_id":null,"source_type":"system","tags":["archival"]},"importance":null,"confidence":null,"created_at":"2026-02-24T19:19:08.696236","updated_at":null,"archived_at":"2026-04-10T13:49:08.703455Z"}



In [6]:
# Warm storage is DB-backed and stores JSON payloads plus metadata.
backend = local_store.warm_layer._backend  # type: ignore[attr-defined]
rows = backend._conn.execute(
    "SELECT id, agent_id, scope, memory_type, layer, content_json, metadata_json FROM memory_records ORDER BY created_at DESC"
).fetchall()
for row in rows:
    print(row)


('981946be-1fff-4481-988b-8e201d48fbbf', 'mailmind', 'agent_local', 'episodic', 'warm', '{"event": "classified a DeepMind recruiting email"}', '{"agent": "mailmind", "agent_id": "mailmind", "confidence": null, "importance": null, "priority": "medium", "scope": "agent_local", "source": "system", "source_id": null, "source_type": "system", "tags": []}')
('e415b19e-bae7-4ff4-9048-e3a736666933', 'mailmind', 'agent_local', 'semantic', 'warm', '{"fact": "user prefers research roles at DeepMind"}', '{"agent": "mailmind", "agent_id": "mailmind", "confidence": null, "importance": null, "priority": "medium", "scope": "agent_local", "source": "system", "source_id": null, "source_type": "system", "tags": ["preference"]}')
('b4e1a80e-ef3c-4792-b785-dd943d9d34ad', 'mailmind', 'agent_local', 'semantic', 'warm', '{"fact": "old archived preference"}', '{"agent": "mailmind", "agent_id": "mailmind", "confidence": null, "importance": null, "priority": "medium", "scope": "agent_local", "source": "system", 

In [7]:
# Retrieval should escalate from local to global when context requires it.
global_record = service.add_global(
    SemanticMemory(content={"fact": "OpenAI endpoint auth uses api keys"}, scope="global")
)

global_results = service.retrieve(
    "api keys",
    filters={"type": "semantic"},
    limit=5,
    context=RetrievalContext(agent_id="mailmind", step_count=4, confidence=0.4, allow_global=True),
)

print({
    "global_id": global_record.id,
    "returned_ids": [record.id for record in global_results],
    "returned_scopes": [record.scope for record in global_results],
})


{'global_id': '5dcea364-3aca-4df6-a6fa-9cf24b6d1906', 'returned_ids': ['5dcea364-3aca-4df6-a6fa-9cf24b6d1906'], 'returned_scopes': ['global']}


In [8]:
# Sleeping tasks are deferred JSONL tasks ordered by priority.
queue = SleepingTaskQueue(MEMORY_ROOT / "sleeping.jsonl")
queue.enqueue(SleepingTask(task_type="reindex", payload={"scope": "agent_local"}, priority=1))
queue.enqueue(SleepingTask(task_type="summarize", payload={"date": "2026-04-09"}, priority=5))

print("queued:", [(task.task_type, task.priority) for task in queue.list_tasks()])
next_task = queue.pop_next()
print("popped:", (next_task.task_type, next_task.priority) if next_task else None)
print("remaining:", [(task.task_type, task.priority) for task in queue.list_tasks()])


queued: [('summarize', 5), ('reindex', 1)]
popped: ('summarize', 5)
remaining: [('reindex', 1)]


In [9]:
# Working memory is in-memory only. Procedural memory is config/code shaped, not runtime DB memory.
working = WorkingMemory(session_id="memory-demo")
for i in range(8):
    working.add_user_message(f"u{i}")
    working.add_agent_message(f"a{i}")
working.set_state(topic="memory-check", selected_tool="memory_search")

procedural = ProceduralMemory(
    tool_names=["EmailSearchTool", "EmailSendTool"],
    planner_names=["mailmind-planner"],
)

print("working_recent:", working.recent_items)
print("working_state:", working.state)
print("procedural:", procedural.model_dump(mode="json"))


working_recent: [{'role': 'user', 'content': 'u5'}, {'role': 'agent', 'content': 'a5'}, {'role': 'user', 'content': 'u6'}, {'role': 'agent', 'content': 'a6'}, {'role': 'user', 'content': 'u7'}, {'role': 'agent', 'content': 'a7'}]
working_state: {'topic': 'memory-check', 'selected_tool': 'memory_search'}
procedural: {'tool_names': ['EmailSearchTool', 'EmailSendTool'], 'planner_names': ['mailmind-planner'], 'llm_names': [], 'prompt_names': []}
